# Notebook 06: EasyEnsemble — RF y XGBoost

**Objetivo**: Entrenar 20 modelos Random Forest y 20 modelos XGBoost usando
EasyEnsemble (undersampling balanceado por iteración) para mejorar la
clasificación de emociones minoritarias (`fear`, `love`, `surprise`).

**Estrategia**:
- En cada iteración se samplea ~4.455 filas por clase del train set
- Subset balanceado: 6 clases × 4.455 ≈ 26.730 filas
- Se repite 20 veces con semillas distintas → 20 modelos RF + 20 modelos XGBoost
- Sobre el test set completo se obtienen probabilidades de cada modelo

**Entrada**: `data/embeddings/train_embeddings.parquet`, `test_embeddings.parquet`  
**Salida**: `results/easy_ensemble_probas.csv`, `models/ee_rf_*.joblib`, `models/ee_xgb_*.joblib`

---

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import joblib
import time
import warnings
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

warnings.filterwarnings('ignore')
print('Imports OK')
print(f'XGBoost version: {xgb.__version__}')


Imports OK
XGBoost version: 1.7.6


## 2. Configuración

In [2]:
# ════════════════════════════════════════════════════════════
# CONFIGURACIÓN — ajustar según necesidad
# ════════════════════════════════════════════════════════════
N_ESTIMATORS   = 20    # iteraciones EasyEnsemble (modelos por tipo)
SAMPLE_SIZE    = 15000 # filas por clase por iteracion (bootstrap si la clase tiene menos)
BASE_SEED      = 42    # semilla base; cada iteración usa BASE_SEED + i
USE_SUBSET     = False # True = usar _subset para pruebas rápidas
# ════════════════════════════════════════════════════════════

suffix = '_subset' if USE_SUBSET else ''

DATA_DIR   = Path('../data/embeddings')
MODELS_DIR = Path('../models')
RESULTS_DIR = Path('../results')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('=' * 60)
print('CONFIGURACIÓN EASYENSEMBLE')
print('=' * 60)
print(f'Iteraciones por modelo : {N_ESTIMATORS}')
print(f'Filas por clase        : {SAMPLE_SIZE}')
print(f'Filas por iteración    : {SAMPLE_SIZE * 6} (6 clases × {SAMPLE_SIZE})')
print(f'Semilla base           : {BASE_SEED}')
print(f'Modo                   : {"SUBSET" if USE_SUBSET else "FULL DATASET"}')


CONFIGURACIÓN EASYENSEMBLE
Iteraciones por modelo : 20
Filas por clase        : 15000
Filas por iteración    : 90000 (6 clases × 15000)
Semilla base           : 42
Modo                   : FULL DATASET


## 3. Carga de Datos

In [3]:
train_df = pd.read_parquet(DATA_DIR / f'train_embeddings{suffix}.parquet')
test_df  = pd.read_parquet(DATA_DIR / f'test_embeddings{suffix}.parquet')

X_train = train_df.drop('emotion', axis=1).values
y_train = train_df['emotion'].values
X_test  = test_df.drop('emotion', axis=1).values
y_test  = test_df['emotion'].values

classes = sorted(np.unique(y_train))
n_classes = len(classes)

print(f'Train : {X_train.shape}')
print(f'Test  : {X_test.shape}')
print(f'Clases: {classes}')
print()
print('Distribución train:')
for cls in classes:
    n = (y_train == cls).sum()
    print(f'  {cls:<12} {n:>7,}  ({n/len(y_train)*100:.1f}%)')
print()
print(f'Filas disponibles por clase vs SAMPLE_SIZE={SAMPLE_SIZE}:')
for cls in classes:
    n = (y_train == cls).sum()
    flag = 'OK' if n >= SAMPLE_SIZE else 'INSUFICIENTE'
    print(f'  {cls:<12} {n:>7,}  {flag}')


Train : (441127, 257)
Test  : (110282, 257)
Clases: ['anger', 'fear', 'joy', 'love', 'sadness', 'surprise']

Distribución train:
  anger         87,742  (19.9%)
  fear          22,478  (5.1%)
  joy          167,205  (37.9%)
  love          22,370  (5.1%)
  sadness      136,858  (31.0%)
  surprise       4,474  (1.0%)

Filas disponibles por clase vs SAMPLE_SIZE=15000:
  anger         87,742  OK
  fear          22,478  OK
  joy          167,205  OK
  love          22,370  OK
  sadness      136,858  OK
  surprise       4,474  INSUFICIENTE


## 4. Helper: Balanced Sampler

En cada iteración se samplea `SAMPLE_SIZE` filas de **cada** clase.
La semilla varía por iteración para garantizar diversidad entre modelos.

In [4]:
def balanced_sample(X, y, sample_size, seed):
    """
    Retorna un subset balanceado con sample_size filas por clase.
    Usa bootstrap (replace=True) si la clase tiene menos filas que sample_size.
    """
    rng = np.random.default_rng(seed)
    idx_list = []
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        use_replace = len(cls_idx) < sample_size
        chosen = rng.choice(cls_idx, size=sample_size, replace=use_replace)
        idx_list.append(chosen)
    idx = np.concatenate(idx_list)
    rng.shuffle(idx)
    return X[idx], y[idx]

# Verificacion
X_s, y_s = balanced_sample(X_train, y_train, SAMPLE_SIZE, BASE_SEED)
print(f'Sample shape: {X_s.shape}')
for cls in classes:
    n = (y_s == cls).sum()
    flag = ' (bootstrap)' if (y_train == cls).sum() < SAMPLE_SIZE else ''
    print(f'  {cls:<12} {n}{flag}')


Sample shape: (90000, 257)
  anger        15000
  fear         15000
  joy          15000
  love         15000
  sadness      15000
  surprise     15000 (bootstrap)


## 5. EasyEnsemble — Random Forest

20 iteraciones. Cada modelo se guarda en `models/ee_rf_{i:02d}.joblib`.
Las probabilidades sobre el test set se acumulan en `rf_probas` (shape: 20 × n_test × 6).

In [5]:
rf_probas = np.zeros((N_ESTIMATORS, len(X_test), n_classes))
rf_models_info = []

print('=' * 60)
print('EASYENSEMBLE — RANDOM FOREST')
print('=' * 60)
t_total = time.time()

for i in range(N_ESTIMATORS):
    seed_i = BASE_SEED + i
    t0 = time.time()

    # 1. Balanced sample
    X_s, y_s = balanced_sample(X_train, y_train, SAMPLE_SIZE, seed_i)

    # 2. Train RF
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=seed_i,
    )
    rf.fit(X_s, y_s)

    # 3. Probabilidades sobre test
    rf_probas[i] = rf.predict_proba(X_test)

    # 4. F1-macro de este modelo solo (orientativo)
    y_pred_i = rf.classes_[rf_probas[i].argmax(axis=1)]
    f1_i = f1_score(y_test, y_pred_i, average='macro', zero_division=0)

    elapsed = time.time() - t0
    print(f'  [{i+1:02d}/{N_ESTIMATORS}] seed={seed_i}  F1-macro={f1_i:.4f}  ({elapsed:.1f}s)')

    # 5. Guardar modelo
    model_path = MODELS_DIR / f'ee_rf_{i:02d}{suffix}.joblib'
    joblib.dump(rf, model_path)
    rf_models_info.append({'iter': i, 'seed': seed_i, 'f1_macro': f1_i, 'path': str(model_path)})

print()
print(f'Tiempo total RF: {time.time() - t_total:.1f}s')

# Ensemble RF: promedio de probabilidades
rf_ensemble_proba = rf_probas.mean(axis=0)
rf_classes = rf.classes_
y_pred_rf_ens = rf_classes[rf_ensemble_proba.argmax(axis=1)]
f1_rf_ens = f1_score(y_test, y_pred_rf_ens, average='macro', zero_division=0)
print(f'F1-macro Ensemble RF (promedio 20 modelos): {f1_rf_ens:.4f}')


EASYENSEMBLE — RANDOM FOREST
  [01/20] seed=42  F1-macro=0.3618  (56.9s)
  [02/20] seed=43  F1-macro=0.3627  (55.5s)
  [03/20] seed=44  F1-macro=0.3654  (56.2s)
  [04/20] seed=45  F1-macro=0.3624  (57.0s)
  [05/20] seed=46  F1-macro=0.3643  (56.7s)
  [06/20] seed=47  F1-macro=0.3631  (58.0s)
  [07/20] seed=48  F1-macro=0.3632  (56.9s)
  [08/20] seed=49  F1-macro=0.3636  (57.0s)
  [09/20] seed=50  F1-macro=0.3637  (55.5s)
  [10/20] seed=51  F1-macro=0.3624  (57.7s)
  [11/20] seed=52  F1-macro=0.3637  (55.8s)
  [12/20] seed=53  F1-macro=0.3649  (57.9s)
  [13/20] seed=54  F1-macro=0.3656  (56.5s)
  [14/20] seed=55  F1-macro=0.3647  (58.5s)
  [15/20] seed=56  F1-macro=0.3629  (58.2s)
  [16/20] seed=57  F1-macro=0.3643  (62.3s)
  [17/20] seed=58  F1-macro=0.3640  (61.2s)
  [18/20] seed=59  F1-macro=0.3613  (57.3s)
  [19/20] seed=60  F1-macro=0.3658  (60.3s)
  [20/20] seed=61  F1-macro=0.3632  (57.2s)

Tiempo total RF: 1160.5s
F1-macro Ensemble RF (promedio 20 modelos): 0.3892


## 6. EasyEnsemble — XGBoost

Igual que RF pero con XGBoost. XGBoost requiere labels numéricos → se usa
un `LabelEncoder` fijo (fiteado sobre todas las clases del train) para
mantener consistencia entre iteraciones.

In [6]:
# LabelEncoder fijo para todas las iteraciones
le = LabelEncoder()
le.fit(y_train)
y_train_enc = le.transform(y_train)
print(f'Clases codificadas: {dict(zip(le.classes_, le.transform(le.classes_)))}')
print()

xgb_probas = np.zeros((N_ESTIMATORS, len(X_test), n_classes))
xgb_models_info = []

print('=' * 60)
print('EASYENSEMBLE — XGBOOST')
print('=' * 60)
t_total = time.time()

for i in range(N_ESTIMATORS):
    seed_i = BASE_SEED + i
    t0 = time.time()

    # 1. Balanced sample (sobre índices, para reusar el encoder)
    rng = np.random.default_rng(seed_i)
    idx_list = []
    for cls_enc in range(n_classes):
        cls_idx = np.where(y_train_enc == cls_enc)[0]
        n = min(SAMPLE_SIZE, len(cls_idx))
        idx_list.append(rng.choice(cls_idx, size=n, replace=False))
    idx = np.concatenate(idx_list)
    rng.shuffle(idx)
    X_s, y_s_enc = X_train[idx], y_train_enc[idx]

    # 2. Train XGBoost
    xgb_model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=seed_i,
        n_jobs=-1,
        verbosity=0,
    )
    xgb_model.fit(X_s, y_s_enc)

    # 3. Probabilidades sobre test (orden: le.classes_)
    xgb_probas[i] = xgb_model.predict_proba(X_test)

    # 4. F1-macro orientativo
    y_pred_enc_i = xgb_model.predict(X_test)
    y_pred_i = le.inverse_transform(y_pred_enc_i)
    f1_i = f1_score(y_test, y_pred_i, average='macro', zero_division=0)

    elapsed = time.time() - t0
    print(f'  [{i+1:02d}/{N_ESTIMATORS}] seed={seed_i}  F1-macro={f1_i:.4f}  ({elapsed:.1f}s)')

    # 5. Guardar modelo
    model_path = MODELS_DIR / f'ee_xgb_{i:02d}{suffix}.joblib'
    joblib.dump(xgb_model, model_path)
    xgb_models_info.append({'iter': i, 'seed': seed_i, 'f1_macro': f1_i, 'path': str(model_path)})

# Guardar LabelEncoder compartido
joblib.dump(le, MODELS_DIR / f'ee_label_encoder{suffix}.joblib')

print()
print(f'Tiempo total XGB: {time.time() - t_total:.1f}s')

# Ensemble XGB
xgb_ensemble_proba = xgb_probas.mean(axis=0)
y_pred_xgb_ens = le.classes_[xgb_ensemble_proba.argmax(axis=1)]
f1_xgb_ens = f1_score(y_test, y_pred_xgb_ens, average='macro', zero_division=0)
print(f'F1-macro Ensemble XGB (promedio 20 modelos): {f1_xgb_ens:.4f}')


Clases codificadas: {'anger': 0, 'fear': 1, 'joy': 2, 'love': 3, 'sadness': 4, 'surprise': 5}

EASYENSEMBLE — XGBOOST
  [01/20] seed=42  F1-macro=0.3877  (148.7s)
  [02/20] seed=43  F1-macro=0.3894  (149.8s)
  [03/20] seed=44  F1-macro=0.3875  (151.2s)
  [04/20] seed=45  F1-macro=0.3880  (154.2s)
  [05/20] seed=46  F1-macro=0.3865  (153.9s)
  [06/20] seed=47  F1-macro=0.3876  (152.1s)
  [07/20] seed=48  F1-macro=0.3892  (149.9s)
  [08/20] seed=49  F1-macro=0.3894  (151.9s)
  [09/20] seed=50  F1-macro=0.3883  (155.5s)
  [10/20] seed=51  F1-macro=0.3877  (158.9s)
  [11/20] seed=52  F1-macro=0.3875  (151.3s)
  [12/20] seed=53  F1-macro=0.3886  (156.1s)
  [13/20] seed=54  F1-macro=0.3870  (157.2s)
  [14/20] seed=55  F1-macro=0.3866  (157.3s)
  [15/20] seed=56  F1-macro=0.3875  (156.3s)
  [16/20] seed=57  F1-macro=0.3877  (155.7s)
  [17/20] seed=58  F1-macro=0.3881  (154.9s)
  [18/20] seed=59  F1-macro=0.3877  (152.4s)
  [19/20] seed=60  F1-macro=0.3875  (150.2s)
  [20/20] seed=61  F1-macro

## 7. Guardar CSV de Probabilidades

Una fila por muestra del test set. Columnas:
- `clase_real`: etiqueta verdadera
- `rf_m{i:02d}_{clase}`: probabilidad del modelo RF i para la clase
- `xgb_m{i:02d}_{clase}`: probabilidad del modelo XGB i para la clase

Total: 1 + 20×6 + 20×6 = **241 columnas**

In [7]:
rows = {'clase_real': y_test}

# RF probas
for i in range(N_ESTIMATORS):
    for j, cls in enumerate(rf_classes):
        rows[f'rf_m{i:02d}_{cls}'] = rf_probas[i, :, j]

# XGB probas
for i in range(N_ESTIMATORS):
    for j, cls in enumerate(le.classes_):
        rows[f'xgb_m{i:02d}_{cls}'] = xgb_probas[i, :, j]

probas_df = pd.DataFrame(rows)

csv_path = RESULTS_DIR / f'easy_ensemble_probas{suffix}.csv'
probas_df.to_csv(csv_path, index=False)

print(f'CSV guardado: {csv_path}')
print(f'Shape: {probas_df.shape}')
print(f'Columnas: {len(probas_df.columns)} (1 target + {N_ESTIMATORS}×6 RF + {N_ESTIMATORS}×6 XGB)')


CSV guardado: ..\results\easy_ensemble_probas.csv
Shape: (110282, 241)
Columnas: 241 (1 target + 20×6 RF + 20×6 XGB)


## 8. Evaluación del Ensemble

Comparamos 5 escenarios:
1. RF Ensemble (promedio 20 RF)
2. XGBoost Ensemble (promedio 20 XGB)
3. Ensemble Completo (promedio 40 modelos RF+XGB)
4. F1 individual por iteración (varianza de modelos)


In [8]:
# Ensemble completo: promedio de los 40 modelos
full_ensemble_proba = (rf_ensemble_proba + xgb_ensemble_proba) / 2
y_pred_full = rf_classes[full_ensemble_proba.argmax(axis=1)]
f1_full = f1_score(y_test, y_pred_full, average='macro', zero_division=0)

print('=' * 60)
print('RESUMEN F1-MACRO')
print('=' * 60)
print(f'RF  Ensemble (20 modelos) : {f1_rf_ens:.4f}')
print(f'XGB Ensemble (20 modelos) : {f1_xgb_ens:.4f}')
print(f'Full Ensemble (40 modelos): {f1_full:.4f}')
print()

# F1 por iteración — varianza
rf_f1s  = [info['f1_macro'] for info in rf_models_info]
xgb_f1s = [info['f1_macro'] for info in xgb_models_info]
print(f'RF  individual — mean={np.mean(rf_f1s):.4f}  std={np.std(rf_f1s):.4f}  min={min(rf_f1s):.4f}  max={max(rf_f1s):.4f}')
print(f'XGB individual — mean={np.mean(xgb_f1s):.4f}  std={np.std(xgb_f1s):.4f}  min={min(xgb_f1s):.4f}  max={max(xgb_f1s):.4f}')


RESUMEN F1-MACRO
RF  Ensemble (20 modelos) : 0.3892
XGB Ensemble (20 modelos) : 0.4002
Full Ensemble (40 modelos): 0.4073

RF  individual — mean=0.3636  std=0.0012  min=0.3613  max=0.3658
XGB individual — mean=0.3879  std=0.0008  min=0.3865  max=0.3894


In [9]:
# Classification report — best ensemble
best_ens_name  = 'Full (RF+XGB)'
best_ens_pred  = y_pred_full
best_ens_f1    = f1_full

if f1_rf_ens > best_ens_f1:
    best_ens_name, best_ens_pred, best_ens_f1 = 'RF Ensemble', y_pred_rf_ens, f1_rf_ens
if f1_xgb_ens > best_ens_f1:
    best_ens_name, best_ens_pred, best_ens_f1 = 'XGB Ensemble', y_pred_xgb_ens, f1_xgb_ens

print(f'Mejor ensemble: {best_ens_name}')
print()
print(classification_report(y_test, best_ens_pred, digits=4, zero_division=0))


Mejor ensemble: Full (RF+XGB)

              precision    recall  f1-score   support

       anger     0.4880    0.6325    0.5509     21936
        fear     0.1905    0.5036    0.2764      5619
         joy     0.7223    0.3798    0.4978     41801
        love     0.1973    0.5864    0.2952      5593
     sadness     0.6254    0.5145    0.5646     34215
    surprise     0.7303    0.1574    0.2590      1118

    accuracy                         0.4864    110282
   macro avg     0.4923    0.4624    0.4073    110282
weighted avg     0.5920    0.4864    0.5051    110282



---

## Resumen

**EasyEnsemble ejecutado**: 20 RF + 20 XGBoost sobre subsets balanceados de ~26.730 filas.

**Artefactos generados**:
- `models/ee_rf_{00..19}.joblib`: 20 modelos Random Forest
- `models/ee_xgb_{00..19}.joblib`: 20 modelos XGBoost
- `models/ee_label_encoder.joblib`: LabelEncoder compartido para XGBoost
- `results/easy_ensemble_probas.csv`: probabilidades de los 40 modelos sobre test set

**Próximo paso**: Notebook 07 — análisis del CSV de probabilidades y evaluación final.